DuckDB 제대로 쓰기

In [38]:
import duckdb
import pandas as pd

In [39]:
# DB 연결
con = duckdb.connect(database='D:/Assets/BinanceFuturesData/binancefuturesdata.duckdb', read_only=True)

In [28]:
# DB 연결 해제
con.close()

In [27]:
# 타임스탬프(ms) -> 날짜
pd.to_datetime(1707801200000, unit='ms')

Timestamp('2024-02-13 05:13:20')

In [26]:
r1 = con.execute("SELECT * FROM symbol where listing_date > 1707801200000").df()
r1.head(20)

,name,liquidation_fee,listing_date,max_price,min_price,tick_size,max_quantity,min_quantity,step_size,price_precision,quantity_precision,underlying_type
0,OMUSDT,0.015,1707832800000,200.0,1.000000e-05,1.000000e-05,9.000000e+06,0.100,0.100,7,1,Coin
1,PIXELUSDT,0.015,1708354800000,200.0,1.000000e-04,1.000000e-05,7.000000e+07,1.000,1.000,7,0,Coin
2,STRKUSDT,0.015,1708448400000,2000.0,1.000000e-04,1.000000e-04,2.000000e+07,0.100,0.100,7,1,Coin
3,GLMUSDT,0.015,1708596000000,200.0,1.000000e-04,1.000000e-05,5.000000e+06,1.000,1.000,7,0,Coin
4,PORTALUSDT,0.025,1709218800000,2000.0,1.000000e-04,1.000000e-05,3.000000e+07,0.100,0.100,7,1,Coin
5,TONUSDT,0.015,1709296200000,200.0,1.000000e-04,1.000000e-04,2.000000e+06,0.100,0.100,7,1,Coin
6,AXLUSDT,0.015,1709307000000,2000.0,1.000000e-04,1.000000e-04,1.000000e+07,0.100,0.100,7,1,Coin
7,MYROUSDT,0.020,1709627400000,200.0,1.000000e-05,1.000000e-05,5.000000e+07,1.000,1.000,7,0,Coin
8,METISUSDT,0.015,1710232200000,2000.0,1.000000e-03,1.000000e-03,8.000000e+04,0.010,0.010,4,2,Coin
9,AEVOUSDT,0.015,1710336600000,2000.0,1.000000e-05,1.000000e-05,2.000000e+07,0.100,0.100,7,1,Coin


In [48]:
query = f"""
SELECT 
    CAST(FLOOR(timestamp / 86400000) * 86400000 AS BIGINT) as aggregated_timestamp,
    FIRST(open ORDER BY timestamp ASC) as open,
    MAX(high) as high,
    MIN(low) as low,
    LAST(close ORDER BY timestamp ASC) as close,
    SUM(volume) as volume,
    SUM(quote_volume) as quote_volume,
    SUM(taker_buy_volume) as taker_buy_volume,
    SUM(taker_buy_quote_volume) as taker_buy_quote_volume,
    SUM(trade_count) as trade_count
FROM quote
WHERE symbol = 'DASHUSDT'
  AND timestamp BETWEEN 0 AND 2000000000000
GROUP BY FLOOR(timestamp / 86400000)
ORDER BY aggregated_timestamp
"""

d1 = con.execute(query).df()
d1.head(30)

,aggregated_timestamp,open,high,low,close,volume,quote_volume,taker_buy_volume,taker_buy_quote_volume,trade_count
0,1580774400000,115.50,115.50,108.47,111.29,26628.215,2.955633e+06,12691.241,1.409114e+06,19902.0
1,1580860800000,111.29,124.76,110.04,122.94,108596.428,1.301939e+07,55477.128,6.654025e+06,31955.0
2,1580947200000,123.09,125.36,117.15,120.87,52416.412,6.375445e+06,22478.341,2.735986e+06,25376.0
3,1581033600000,120.84,123.38,118.26,118.88,50706.740,6.096321e+06,23500.874,2.826417e+06,26414.0
4,1581120000000,118.88,132.32,113.16,126.64,150490.454,1.896110e+07,74452.732,9.389555e+06,66591.0
5,1581206400000,126.56,131.41,124.00,128.67,58992.758,7.566165e+06,27012.517,3.467590e+06,28307.0
6,1581292800000,128.73,129.18,122.95,127.84,47034.188,5.947792e+06,20835.001,2.635497e+06,29290.0
7,1581379200000,127.88,134.30,124.14,129.83,79948.978,1.029355e+07,38940.943,5.020032e+06,46040.0
8,1581465600000,129.93,137.67,128.94,133.70,94220.558,1.261593e+07,43527.592,5.826924e+06,51554.0
9,1581552000000,133.70,138.74,127.00,131.17,106069.328,1.401646e+07,47768.827,6.318410e+06,53823.0


In [42]:
r2 = con.execute("select * from quote where symbol = 'ENSOUSDT' order by timestamp asc limit 1000").df()
r2

,symbol,interval,timestamp,open,high,low,close,volume,quote_volume,taker_buy_volume,taker_buy_quote_volume,trade_count
0,ENSOUSDT,1,1760432400000,3.342,3.342,0.001,2.741,1846935.3,5.084057e+06,987494.1,2.648764e+06,16689
1,ENSOUSDT,1,1760432460000,2.737,3.149,2.727,2.954,1681614.2,5.032093e+06,827574.8,2.476023e+06,21548
2,ENSOUSDT,1,1760432520000,2.951,2.975,2.821,2.879,1036281.6,3.000794e+06,547592.1,1.585290e+06,17138
3,ENSOUSDT,1,1760432580000,2.879,3.153,2.878,3.053,1276491.2,3.859311e+06,721687.2,2.179803e+06,24779
4,ENSOUSDT,1,1760432640000,3.054,3.292,3.018,3.110,1338301.8,4.249528e+06,747653.9,2.375381e+06,22557
...,...,...,...,...,...,...,...,...,...,...,...,...
995,ENSOUSDT,1,1760492100000,2.993,2.994,2.820,2.881,300601.2,8.646019e+05,100548.2,2.887551e+05,8165
996,ENSOUSDT,1,1760492160000,2.881,2.908,2.843,2.859,116448.4,3.346506e+05,43388.0,1.249252e+05,3129
997,ENSOUSDT,1,1760492220000,2.859,2.865,2.517,2.599,1082555.5,2.859321e+06,400821.8,1.054326e+06,19600
998,ENSOUSDT,1,1760492280000,2.597,2.640,2.585,2.625,181798.7,4.752203e+05,91567.0,2.394193e+05,3791


In [5]:
r3 = con.execute("PRAGMA table_info(quote)").df()
r3

,cid,name,type,notnull,dflt_value,pk
0,0,symbol,VARCHAR,True,None,True
1,1,interval,TINYINT,True,None,True
2,2,timestamp,BIGINT,True,None,True
3,3,open,"DECIMAL(18,8)",False,None,False
4,4,high,"DECIMAL(18,8)",False,None,False
5,5,low,"DECIMAL(18,8)",False,None,False
6,6,close,"DECIMAL(18,8)",False,None,False
7,7,volume,"DECIMAL(24,8)",False,None,False
8,8,quote_volume,"DECIMAL(18,8)",False,None,False
9,9,taker_buy_volume,"DECIMAL(24,8)",False,None,False


In [ ]:
query = """
WITH daily_buckets AS (
    SELECT 
        symbol,
        FLOOR(timestamp / 86400000) AS day_bucket,  -- 86400000 ms = 1일
        MAX(high)  AS daily_high,
        MIN(low)   AS daily_low
    FROM quote
    WHERE interval = 1 
      AND low > 0
    GROUP BY symbol, day_bucket
    HAVING daily_low > 0
),
with_row_num AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY day_bucket) AS rn
    FROM daily_buckets
),
filtered_days AS (
    SELECT 
        symbol,
        day_bucket,
        (daily_high - daily_low) / daily_low AS daily_vol_ratio
    FROM with_row_num
    WHERE rn >= 3
),
aggregated AS (
    SELECT 
        symbol,
        ROUND(MAX(daily_vol_ratio) * 100, 2) AS max_daily_vol_pct,
        ROUND(MIN(daily_vol_ratio) * 100, 2) AS min_daily_vol_pct,
        COUNT(*) AS num_days_calculated,
        ROUND(AVG(daily_vol_ratio) * 100, 2) AS avg_daily_vol_pct,
        
        -- 상장일자 계산: symbol별 가장 이른 day_bucket (첫 데이터 날)
        MIN(day_bucket) AS first_day_bucket
    FROM filtered_days
    GROUP BY symbol
    HAVING COUNT(*) >= 1
)
SELECT 
    symbol,
    -- 상장일자: ms timestamp → 초 → DuckDB timestamp → 날짜 문자열로 변환
    strftime(
        to_timestamp(first_day_bucket * 86400000 / 1000.0),
        '%Y-%m-%d'
    ) AS listing_date,
    
    max_daily_vol_pct,
    min_daily_vol_pct,
    num_days_calculated,
    avg_daily_vol_pct
FROM aggregated
WHERE max_daily_vol_pct < 60
ORDER BY max_daily_vol_pct DESC;
"""

r4 = con.execute(query).df()
r4

,symbol,max_daily_vol_pct,min_daily_vol_pct,num_days_calculated,avg_daily_vol_pct
0,GLMUSDT,59.20,1.74,684,8.70
1,NIGHTUSDT,59.15,6.38,27,18.78
2,TWTUSDT,59.08,1.22,795,6.92
3,ONUSDT,59.07,1.33,74,12.84
4,TUSDT,58.78,1.39,1070,7.65
5,QNTUSDT,56.17,0.77,1174,6.52
6,PUNDIXUSDT,55.78,1.68,251,7.38
7,RIFUSDT,55.61,1.62,808,8.65
8,TURTLEUSDT,55.43,3.39,76,12.87
9,ZKPUSDT,55.09,5.74,16,19.28


In [ ]:
#1. 전체 행 수와 NULL 개수 한눈에 보기
result = con.sql("""
                 SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(open)   AS null_open,
    COUNT(*) - COUNT(high)   AS null_high,
    COUNT(*) - COUNT(low)    AS null_low,
    COUNT(*) - COUNT(close)  AS null_close,
    COUNT(*) - COUNT(volume) AS null_volume,
    COUNT(*) - COUNT(timestamp) AS null_timestamp  -- 이건 거의 0이어야 함
FROM quote;   -- ← 실제 테이블명으로 변경
                 """).df()
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,null_open,null_high,null_low,null_close,null_volume,null_timestamp
0,651762541,0,0,0,0,0,0


In [6]:
#2. symbol + interval 별로 데이터가 얼마나 깨졌는지 (가장 유용!)
result = con.sql("""
                 SELECT 
    symbol,
    COUNT(*) AS row_count,
    COUNT(*) FILTER(WHERE open IS NULL OR close IS NULL) AS null_ohlc_count,
    MIN(timestamp) AS first_ts,
    MAX(timestamp) AS last_ts,
    (MAX(timestamp) - MIN(timestamp)) / 1000 / 60 AS total_minutes_span  -- 분 단위
FROM quote
GROUP BY symbol
ORDER BY null_ohlc_count DESC, row_count DESC;
                 """).df()
result

,symbol,row_count,null_ohlc_count,first_ts,last_ts,total_minutes_span
0,BTCUSDT,3330237,0,1567969800000,1767783960000,3330236.0
1,ETHUSDT,3215722,0,1574840700000,1767783960000,3215721.0
2,BCHUSDT,3183970,0,1576745820000,1767783960000,3183969.0
3,XRPUSDT,3158086,0,1578298860000,1767783960000,3158085.0
4,LTCUSDT,3153778,0,1578557340000,1767783960000,3153777.0
...,...,...,...,...,...,...
607,IRUSDT,24525,0,1766313000000,1767784440000,24524.0
608,BREVUSDT,11550,0,1767091500000,1767784440000,11549.0
609,COLLECTUSDT,9960,0,1767186900000,1767784440000,9959.0
610,MAGMAUSDT,9945,0,1767187800000,1767784440000,9944.0


In [ ]:
# 변동률 임계값 설정 (예: 0.05 = 5%)
threshold = 1.0

# 1.0 으로 조회시 나온 심볼
# AERGOUSDT
# BULLAUSDT
# CVXUSDT
# CYBERUSDT
# FORMUSDT
# INUSDT
# LITUSDT
# PLUMEUSDT

query = f"""
WITH price_diffs AS (
    SELECT 
        symbol,
        timestamp,
        close,
        LAG(close) OVER (PARTITION BY symbol ORDER BY timestamp) as prev_close
    FROM quote
)
SELECT 
    symbol,
    timestamp,
    prev_close,
    close,
    ABS(close - prev_close) / prev_close as change_rate
FROM price_diffs
WHERE ABS(close - prev_close) / prev_close > {threshold}
ORDER BY symbol, timestamp;
"""

spikes_df = con.execute(query).df()
spikes_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,symbol,timestamp,prev_close,close,change_rate
0,AERGOUSDT,1744801200000,0.06790,0.35258,4.192636
1,BULLAUSDT,1760131440000,0.02298,0.04904,1.134030
2,CVXUSDT,1668366600000,3.87300,8.76500,1.263104
3,CVXUSDT,1753270200000,2.37400,4.97000,1.093513
4,CYBERUSDT,1754983800000,1.88000,4.47300,1.379255
5,FORMUSDT,1760131620000,0.21230,0.44220,1.082902
6,INUSDT,1760076000000,0.12252,0.25949,1.117940
7,LITUSDT,1766511000000,0.59200,3.87900,5.552365
8,PLUMEUSDT,1760131380000,0.01510,0.03245,1.149007
